In [26]:
import pandas as pd
import numpy as np
import pyarrow

print("Ready")

Ready


In [27]:
from pathlib import Path

DATA_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\Data_set")

In [28]:
files = sorted(DATA_PATH.glob("*.csv"))

print("Total CSV files:", len(files))
print("Sample files:")
for f in files[:5]:
    print(f.name)

Total CSV files: 480
Sample files:
WE__0300___________DER_ANCILLARY___________P01.csv
WE__0300___________DER_PS__________________P01.csv
WE__0300___________DER_RHS_________________P01.csv
WE__0300___________DER_TIRS________________P02.csv
WE__0300___________DER_WS__________________P01.csv


In [29]:
import re

sols = []

for f in files:
    match = re.search(r"WE__(\d{4})", f.name)
    if match:
        sols.append(int(match.group(1)))

print("Total sols:", len(set(sols)))
print("Sol range:", min(sols), "→", max(sols))

Total sols: 116
Sol range: 300 → 419


In [30]:
# STEP 1 — PER-SOL MERGE + PARQUET

In [31]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from collections import defaultdict
from tqdm import tqdm

DATA_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\Data_set")
OUT_PATH  = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_parquet")
OUT_PATH.mkdir(exist_ok=True)

REQUIRED_TIME_COLS = ["sclk", "lmst", "ltst"]

In [32]:
# -------------------------
# Standardize columns
# -------------------------
def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )
    return df

In [33]:
# -------------------------
# Load CSV safely
# -------------------------
def load_csv(path):
    df = pd.read_csv(path)
    df = standardize_columns(df)
    df = df.drop_duplicates(subset=REQUIRED_TIME_COLS)
    return df

In [34]:
# -------------------------
# Prefix sensor columns
# -------------------------
def prefix_cols(df, prefix):
    return df.rename(columns={
        c: f"{prefix}_{c}" for c in df.columns if c not in REQUIRED_TIME_COLS
    })

In [35]:
# -------------------------
# Group files by sol
# -------------------------
files = sorted(DATA_PATH.glob("*.csv"))
sol_files = defaultdict(dict)

for file_path in files:
    name = file_path.name

    sol_match = re.search(r"WE__(\d{4})", name)
    if not sol_match:
        continue

    sol = int(sol_match.group(1))

    if "ANCILLARY" in name:
        sol_files[sol]["anc"] = file_path
    elif "DER_PS" in name:
        sol_files[sol]["ps"] = file_path
    elif "DER_RHS" in name:
        sol_files[sol]["rhs"] = file_path
    elif "DER_TIRS" in name:
        sol_files[sol]["tirs"] = file_path

In [36]:
# -------------------------
# Filter complete sols
# -------------------------
complete_sols = [
    sol for sol, sensors in sol_files.items()
    if all(k in sensors for k in ["anc", "ps", "rhs", "tirs"])
]

print("Complete sols:", len(complete_sols))

Complete sols: 116


In [37]:
# =========================
# PER-SOL MERGE + SAVE
# =========================

for i, sol in enumerate(tqdm(sorted(complete_sols))):

    try:
        files = sol_files[sol]

        # -------------------------
        # LOAD DATA
        # -------------------------
        anc_raw = load_csv(files["anc"])

        # PRINT ONCE TO FIND LS COLUMN
        if i == 0:
            print("\n===== ANC COLUMNS =====")
            print(anc_raw.columns.tolist())

        anc  = prefix_cols(anc_raw, "anc")
        ps   = prefix_cols(load_csv(files["ps"]), "ps")
        rhs  = prefix_cols(load_csv(files["rhs"]), "rhs")
        tirs = prefix_cols(load_csv(files["tirs"]), "tirs")

        # -------------------------
        # AUTO-DETECT LS COLUMN
        # -------------------------
        ls_candidates = [c for c in anc.columns if "solar" in c or "ls" in c]

        if i == 0:
            print("\n===== LS CANDIDATES =====")
            print(ls_candidates)

        if len(ls_candidates) == 0:
            raise ValueError(f"❌ No Ls column found in sol {sol}")

        LS_COL = ls_candidates[0]  # pick first match

        # -------------------------
        # MERGE
        # -------------------------
        df_sol = anc.merge(ps, on=REQUIRED_TIME_COLS, how="left")
        df_sol = df_sol.merge(rhs, on=REQUIRED_TIME_COLS, how="left")
        df_sol = df_sol.merge(tirs, on=REQUIRED_TIME_COLS, how="left")

        df_sol["sol_id"] = sol

        # -------------------------
        # TIME RECONSTRUCTION
        # -------------------------
        df_sol["sol"] = df_sol["lmst"].str.extract(r"(\d+)M").astype("Int64")

        df_sol["lmst_time_str"] = df_sol["lmst"].str.extract(r"M(.+)")

        df_sol["lmst_time"] = pd.to_datetime(
            df_sol["lmst_time_str"],
            format="%H:%M:%S.%f",
            errors="coerce"
        )

        df_sol["seconds_in_sol"] = (
            df_sol["lmst_time"].dt.hour * 3600
            + df_sol["lmst_time"].dt.minute * 60
            + df_sol["lmst_time"].dt.second
            + df_sol["lmst_time"].dt.microsecond / 1e6
        )

        MARS_SOL_SECONDS = 88775.244

        df_sol["continuous_time"] = (
            df_sol["sol"].astype(float) * MARS_SOL_SECONDS
            + df_sol["seconds_in_sol"]
        )

        df_sol = df_sol.sort_values("continuous_time")

        # -------------------------
        # KEEP IMPORTANT COLS (SAFE)
        # -------------------------
        keep_cols = [
            "sclk",
            "lmst",
            "ltst",
            "continuous_time",
            "ps_pressure",
            "rhs_local_relative_humidity",
            "rhs_humidity_local_temp",
            LS_COL   # dynamically detected
        ]

        missing_cols = [c for c in keep_cols if c not in df_sol.columns]

        if missing_cols:
            print(f"Missing in sol {sol}: {missing_cols}")

        df_sol = df_sol[[c for c in keep_cols if c in df_sol.columns]]

        # -------------------------
        # RENAME LS TO STANDARD NAME
        # -------------------------
        df_sol = df_sol.rename(columns={LS_COL: "ls_deg"})

        # -------------------------
        # SAVE
        # -------------------------
        out_file = OUT_PATH / f"sol_{sol}.parquet"
        df_sol.to_parquet(out_file, index=False)

        # free memory
        del df_sol

    except Exception as e:
        print(f"Error in sol {sol}: {e}")

  0%|          | 0/116 [00:00<?, ?it/s]


===== ANC COLUMNS =====
['sclk', 'lmst', 'ltst', 'solar_longitude_angle', 'solar_zenithal_angle', 'rover_position_x', 'rover_position_y', 'rover_position_z', 'rover_velocity', 'rover_pitch', 'rover_yaw', 'rover_roll', 'masthead_azimuth', 'masthead_elevation']

===== LS CANDIDATES =====
['anc_solar_longitude_angle', 'anc_solar_zenithal_angle']


100%|██████████| 116/116 [02:29<00:00,  1.29s/it]


In [38]:
files = list(OUT_PATH.glob("*.parquet"))
df = pd.read_parquet(files[0])

print(df.columns)

Index(['sclk', 'lmst', 'ltst', 'continuous_time', 'ps_pressure',
       'rhs_local_relative_humidity', 'rhs_humidity_local_temp', 'ls_deg'],
      dtype='object')


In [2]:
# Sanity Check

from pathlib import Path
import pandas as pd

PARQUET_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_parquet")

files = sorted(PARQUET_PATH.glob("*.parquet"))

print("Total parquet files:", len(files))

Total parquet files: 116


In [3]:
all_columns = []

for f in files:
    df = pd.read_parquet(f)
    all_columns.append(tuple(df.columns))

unique_columns = set(all_columns)

print("Unique column structures:", len(unique_columns))

for cols in unique_columns:
    print(cols)

Unique column structures: 1
('sclk', 'lmst', 'ltst', 'continuous_time', 'ps_pressure', 'rhs_local_relative_humidity', 'rhs_humidity_local_temp', 'ls_deg')


In [4]:
# =========================
# CHECK LMST QUALITY
# =========================

# Total rows
total_rows = len(df)

# Missing lmst
missing_lmst = df["lmst"].isna().sum()

# Invalid format (failed extraction)
lmst_time_str = df["lmst"].str.extract(r"M(.+)")
invalid_format = lmst_time_str.isna().sum()

print("Total rows:", total_rows)
print("Missing lmst:", missing_lmst)
print("Invalid lmst format:", invalid_format)

print("\nPercent missing:", missing_lmst / total_rows)
print("Percent invalid format:", invalid_format / total_rows)

Total rows: 78187
Missing lmst: 0
Invalid lmst format: 0    0
dtype: int64

Percent missing: 0.0
Percent invalid format: 0    0.0
dtype: float64


In [5]:
# STEP 2 — COMBINE + RESAMPLE + PHYSICS FEATURES

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

PARQUET_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_parquet")
OUT_FINAL = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final")
OUT_FINAL.mkdir(exist_ok=True)

files = sorted(PARQUET_PATH.glob("*.parquet"))

In [7]:
# =========================
# 1. COMBINE ALL SOLS
# =========================
df_list = [pd.read_parquet(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

df = df.sort_values("continuous_time").reset_index(drop=True)

print("Combined shape:", df.shape)
print("Time monotonic:", df["continuous_time"].is_monotonic_increasing)
print("Duplicates:", df.duplicated("continuous_time").sum())

Combined shape: (6572418, 8)
Time monotonic: True
Duplicates: 0


In [8]:
# =========================
# 2. FIX Ls COLUMN
# =========================
ls_candidates = [c for c in df.columns if "solar" in c.lower() or "ls" in c.lower()]

if len(ls_candidates) == 0:
    raise ValueError("No Ls column found")

LS_COL = ls_candidates[0]
df["ls_deg"] = pd.to_numeric(df[LS_COL], errors="coerce") % 360

print("Using LS column:", LS_COL)

Using LS column: ls_deg


In [9]:
# TIME RECONSTRUCTION (PHYSICS)

In [10]:
# =========================
# 3. TIME RECONSTRUCTION (PHYSICS-CONSISTENT)
# =========================

import numpy as np
import pandas as pd

# -------------------------
# CONSTANTS
# -------------------------
SOL_SECONDS = 88775.244
SOL_HOURS = 24.6597

# -------------------------
# EXTRACT SOL
# -------------------------
df["sol"] = df["lmst"].str.extract(r"(\d+)M").astype(float)

# -------------------------
# PARSE LMST TIME
# -------------------------
df["lmst_time_str"] = df["lmst"].str.extract(r"M(.+)")
df["lmst_time"] = pd.to_datetime(
    df["lmst_time_str"],
    format="%H:%M:%S.%f",
    errors="coerce"
)

# -------------------------
# LMST AS 24-HOUR MARS CLOCK
# -------------------------
df["hour_of_sol_mars"] = (
    df["lmst_time"].dt.hour
    + df["lmst_time"].dt.minute / 60.0
    + df["lmst_time"].dt.second / 3600.0
    + df["lmst_time"].dt.microsecond / 3.6e9
)

# -------------------------
# FRACTION OF SOL (MASTER TIME VARIABLE)
# -------------------------
df["frac_sol"] = df["hour_of_sol_mars"] / 24.0

# Optional raw physical versions on original dataframe
df["hour_of_sol"] = df["frac_sol"] * SOL_HOURS
df["seconds_in_sol"] = df["frac_sol"] * SOL_SECONDS

In [11]:
# =========================
# 4. GAP DETECTION (SEGMENT SAFE)
# =========================

dt = df["continuous_time"].diff()
median_dt = dt.median()

SMALL_GAP = 10 * median_dt
LARGE_GAP = 60 * median_dt

# Use LARGE_GAP here for real segmentation
df["is_large_gap"] = (dt > LARGE_GAP).astype(int)
df["segment_id"] = df["is_large_gap"].cumsum()

print("\nMedian dt:", median_dt)
print("Small gap threshold:", SMALL_GAP)
print("Large gap threshold:", LARGE_GAP)
print("Segments:", df["segment_id"].nunique())


Median dt: 0.9730000011622906
Small gap threshold: 9.730000011622906
Large gap threshold: 58.380000069737434
Segments: 1101


In [12]:
# =========================
# 5. RESAMPLING (SEGMENT-WISE SAFE)
# =========================

value_cols = [
    "ps_pressure",
    "rhs_local_relative_humidity",
    "rhs_humidity_local_temp"
]

resampled_segments = []

for seg_id, g in df.groupby("segment_id"):

    g = g.copy().sort_values("continuous_time")

    if g.empty:
        continue

    # Snap to 1-min grid
    g["ct_1min"] = (g["continuous_time"] // 60) * 60

    # Aggregate within each minute
    g_val  = g.groupby("ct_1min")[value_cols].mean()
    g_ls   = g.groupby("ct_1min")["ls_deg"].first()
    g_frac = g.groupby("ct_1min")["frac_sol"].first()

    g_grouped = pd.concat([g_val, g_ls, g_frac], axis=1).reset_index()

    if g_grouped.empty:
        continue

    # Build continuous grid inside each segment only
    start_t = g_grouped["ct_1min"].min()
    end_t   = g_grouped["ct_1min"].max()

    grid = pd.DataFrame({
        "ct_1min": np.arange(start_t, end_t + 60, 60.0)
    })

    merged = grid.merge(g_grouped, on="ct_1min", how="left")
    merged = merged.rename(columns={"ct_1min": "continuous_time"})
    merged["segment_id"] = seg_id

    resampled_segments.append(merged)

In [13]:
# =========================
# 6. COMBINE SEGMENTS
# =========================

df_1min = pd.concat(resampled_segments, ignore_index=True)
df_1min = df_1min.sort_values("continuous_time").reset_index(drop=True)

# Reconstruct physical time ONLY AFTER resampling
df_1min["hour_of_sol"] = df_1min["frac_sol"] * SOL_HOURS
df_1min["seconds_in_sol"] = df_1min["frac_sol"] * SOL_SECONDS

In [14]:
# =========================
# 7. DIURNAL FEATURES (CORRECT)
# =========================

angle_hour = 2 * np.pi * df_1min["frac_sol"]

df_1min["sin_hour"] = np.sin(angle_hour)
df_1min["cos_hour"] = np.cos(angle_hour)

In [15]:
# =========================
# 7B. PHYSICS-BASED DAY/NIGHT
# =========================

LATITUDE = np.deg2rad(18.4)   # Jezero crater latitude
OBLIQUITY = np.deg2rad(25.19) # Mars axial tilt

Ls_rad = np.deg2rad(df_1min["ls_deg"])

delta = np.arcsin(
    np.sin(OBLIQUITY) * np.sin(Ls_rad)
)

# Hour angle from fraction of sol
H = 2 * np.pi * (df_1min["frac_sol"] - 0.5)

sin_elevation = (
    np.sin(LATITUDE) * np.sin(delta)
    + np.cos(LATITUDE) * np.cos(delta) * np.cos(H)
)

df_1min["is_day"] = (sin_elevation > 0).astype(int)

In [16]:
# =========================
# 8. SEASONAL FEATURES
# =========================

angle_ls = 2 * np.pi * (df_1min["ls_deg"] / 360.0)

df_1min["sin_ls"] = np.sin(angle_ls)
df_1min["cos_ls"] = np.cos(angle_ls)

In [17]:
# =========================
# 9. SIGNAL CHECK
# =========================

print("\n===== SIGNAL CHECK =====")
print("Pressure STD:", df_1min["ps_pressure"].std())
print("Temp STD:", df_1min["rhs_humidity_local_temp"].std())


===== SIGNAL CHECK =====
Pressure STD: 32.8683480142121
Temp STD: 23.200826462002993


In [18]:
# =========================
# 10. FINAL STRUCTURE
# =========================

final_cols = [
    "continuous_time",
    "segment_id",

    "ps_pressure",
    "rhs_local_relative_humidity",
    "rhs_humidity_local_temp",

    "seconds_in_sol",
    "hour_of_sol",
    "sin_hour",
    "cos_hour",
    "is_day",

    "ls_deg",
    "sin_ls",
    "cos_ls"
]

df_1min = df_1min[final_cols]

In [19]:
# =========================
# 11. SANITY CHECK
# =========================

print("\nFinal shape:", df_1min.shape)

print("\nMissing ratio:")
print(df_1min.isna().mean())

print("\nTime monotonic:", df_1min["continuous_time"].is_monotonic_increasing)

print("\nHour range:", df_1min["hour_of_sol"].min(), df_1min["hour_of_sol"].max())
print("Seconds range:", df_1min["seconds_in_sol"].min(), df_1min["seconds_in_sol"].max())
print("Day ratio:", df_1min["is_day"].mean())


Final shape: (107678, 13)

Missing ratio:
continuous_time                0.000000
segment_id                     0.000000
ps_pressure                    0.006120
rhs_local_relative_humidity    0.109131
rhs_humidity_local_temp        0.108899
seconds_in_sol                 0.000000
hour_of_sol                    0.000000
sin_hour                       0.000000
cos_hour                       0.000000
is_day                         0.000000
ls_deg                         0.000000
sin_ls                         0.000000
cos_ls                         0.000000
dtype: float64

Time monotonic: True

Hour range: 8.27698263888889e-06 24.659638636163198
Seconds range: 0.029797246250000003 88775.02308938127
Day ratio: 0.4980311669979012


In [20]:
# =========================
# 12. SAVE
# =========================

final_path = OUT_FINAL / "meda_1min_FINAL_PHYSICS.parquet"
df_1min.to_parquet(final_path, index=False)

print("\nSaved:", final_path)


Saved: C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_1min_FINAL_PHYSICS.parquet


In [21]:
# =========================
# STEP 2 — Ls VALIDATION
# =========================

In [22]:
import numpy as np
from pathlib import Path

OUT_FINAL = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final")

# -------------------------
# 1. RANGE CHECK (0–360)
# -------------------------
print("\n===== LS RANGE CHECK =====")

print("Min Ls:", df_1min["ls_deg"].min())
print("Max Ls:", df_1min["ls_deg"].max())

invalid_low = (df_1min["ls_deg"] < 0).sum()
invalid_high = (df_1min["ls_deg"] > 360).sum()

print("Values < 0:", invalid_low)
print("Values > 360:", invalid_high)

# Strict validation
if invalid_low > 0 or invalid_high > 0:
    print("ERROR: Ls out of bounds!")
else:
    print("Ls within valid range (0–360)")


===== LS RANGE CHECK =====
Min Ls: 0.02
Max Ls: 359.99
Values < 0: 0
Values > 360: 0
Ls within valid range (0–360)


In [23]:
# -------------------------
# 2. WRAP-AROUND CHECK
# -------------------------
print("\n===== WRAP CHECK =====")

ls_diff = df_1min["ls_deg"].diff()

large_jumps = ls_diff.abs() > 300  # wrap threshold

print("Wrap occurrences:", large_jumps.sum())

print("\nSample wrap points:")
print(df_1min.loc[large_jumps, ["ls_deg"]].head())


===== WRAP CHECK =====
Wrap occurrences: 100

Sample wrap points:
      ls_deg
892   345.65
1602  359.85
2483  345.34
3298  359.78
4079  345.01


In [24]:
# -------------------------
# 3. DISTRIBUTION CHECK
# -------------------------
print("\n===== DISTRIBUTION CHECK =====")

print(df_1min["ls_deg"].describe())

# Optional histogram bins (quick insight)
hist_counts = np.histogram(df_1min["ls_deg"], bins=12)[0]
print("Histogram (12 bins):", hist_counts)


===== DISTRIBUTION CHECK =====
count    107678.000000
mean        177.299373
std          97.935790
min           0.020000
25%          88.120000
50%         159.180000
75%         271.080000
max         359.990000
Name: ls_deg, dtype: float64
Histogram (12 bins): [ 3490  5958 19161 18534  5672  2927  2529  4999 16440 18947  5690  3331]


In [25]:
# -------------------------
# 4. CYCLICAL CONSISTENCY (FINAL CHECK)
# -------------------------
print("\n===== CYCLICAL CONSISTENCY =====")

circle_error = (df_1min["sin_ls"]**2 + df_1min["cos_ls"]**2 - 1).abs().mean()
print("Circle error:", circle_error)

if circle_error < 1e-6:
    print("Cyclical encoding correct")
else:
    print("Problem in sin/cos encoding")


===== CYCLICAL CONSISTENCY =====
Circle error: 3.281033924257669e-17
Cyclical encoding correct


In [26]:
# -------------------------
# 5. FINAL STATUS
# -------------------------
print("\n===== FINAL Ls STATUS =====")

if (
    invalid_low == 0 and
    invalid_high == 0 and
    circle_error < 1e-6
):
    print("Ls VALIDATED — SAFE FOR MODELING")
else:
    print("Ls NOT SAFE — FIX BEFORE CONTINUE")


===== FINAL Ls STATUS =====
Ls VALIDATED — SAFE FOR MODELING


In [27]:
# -------------------------
# 6. SAVE FINAL DATASET
# -------------------------
final_path = OUT_FINAL / "meda_1min_final_with_ls.parquet"

df_1min.to_parquet(final_path, index=False)

print("\nSaved file:")
print(final_path)

print("Final shape:", df_1min.shape)


Saved file:
C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_1min_final_with_ls.parquet
Final shape: (107678, 13)


In [28]:
# =========================
# Sanity DATA CHECK
# =========================

print("===== BASIC INFO =====")
print("Shape:", df_1min.shape)

print("\n===== COLUMNS =====")
print(df_1min.columns.tolist())

print("\n===== SAMPLE VALUES =====")
print(df_1min.head())

print("\n===== NUMERIC SUMMARY =====")
print(df_1min.describe())

print("\n===== MISSING VALUES =====")
print(df_1min.isna().sum())

print("\n===== UNIQUE CHECK (LOW-CARDINALITY) =====")
for col in df_1min.columns:
    unique_vals = df_1min[col].nunique()
    if unique_vals < 10:
        print(f"{col}: {unique_vals} unique values")
        print(df_1min[col].unique())

===== BASIC INFO =====
Shape: (107678, 13)

===== COLUMNS =====
['continuous_time', 'segment_id', 'ps_pressure', 'rhs_local_relative_humidity', 'rhs_humidity_local_temp', 'seconds_in_sol', 'hour_of_sol', 'sin_hour', 'cos_hour', 'is_day', 'ls_deg', 'sin_ls', 'cos_ls']

===== SAMPLE VALUES =====
   continuous_time  segment_id  ps_pressure  rhs_local_relative_humidity  \
0       26632560.0           0   633.009375                     5.763958   
1       26632620.0           0   633.048689                     5.806066   
2       26632680.0           0   633.145000                     5.800645   
3       26632740.0           0   633.204355                     5.740806   
4       26632800.0           0   633.249508                     5.821148   

   rhs_humidity_local_temp  seconds_in_sol  hour_of_sol  sin_hour  cos_hour  \
0               208.745833        0.735684     0.000204  0.000052  1.000000   
1               208.694918       48.735965     0.013538  0.003449  0.999994   
2          

In [29]:
# =========================
# STEP 3 — QC + MISSING HANDLING
# =========================

In [30]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# LOAD STEP 2 OUTPUT
# =========================
DATA_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_1min_FINAL_PHYSICS.parquet")
OUT_PATH  = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_clean_STEP3.parquet")

df = pd.read_parquet(DATA_PATH)

print("Loaded shape:", df.shape)

Loaded shape: (107678, 13)


In [31]:
# =========================
# 1. PHYSICS QC
# =========================

# Pressure (Mars ~600–750 Pa typical, allow safe range)
df.loc[(df["ps_pressure"] < 400) | (df["ps_pressure"] > 1200), "ps_pressure"] = np.nan

# Temperature (Kelvin)
df.loc[(df["rhs_humidity_local_temp"] < 130) | (df["rhs_humidity_local_temp"] > 320),
       "rhs_humidity_local_temp"] = np.nan

# Relative Humidity
df.loc[(df["rhs_local_relative_humidity"] < 0) |
       (df["rhs_local_relative_humidity"] > 100),
       "rhs_local_relative_humidity"] = np.nan

print("\nQC applied")


QC applied


In [32]:
# =========================
# 2. CREATE MISSING FLAGS
# =========================

value_cols = [
    "ps_pressure",
    "rhs_local_relative_humidity",
    "rhs_humidity_local_temp"
]

for col in value_cols:
    df[f"{col}_was_missing"] = df[col].isna().astype(int)

print("Missing flags created")

Missing flags created


In [33]:
# =========================
# 3. GAP-AWARE INTERPOLATION
# =========================

def interpolate_segment(g):
    g = g.copy()
    
    for col in value_cols:
        g[col] = g[col].interpolate(limit=5) 
    
    return g

df = df.groupby("segment_id", group_keys=False).apply(interpolate_segment)

print("Interpolation done (segment-safe)")

Interpolation done (segment-safe)


C:\Users\thari\AppData\Local\Temp\ipykernel_33368\3538448163.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("segment_id", group_keys=False).apply(interpolate_segment)


In [34]:
# =========================
# 4. LIMITED FILLING (PHYSICS-AWARE)
# =========================

# Pressure → safe to carry forward/backward short gaps
df["ps_pressure"] = df["ps_pressure"].ffill(limit=3).bfill(limit=3)

# Temperature → moderate fill
df["rhs_humidity_local_temp"] = (
    df["rhs_humidity_local_temp"].ffill(limit=2).bfill(limit=2)
)

# RH → Mars is dry → safe fallback
df["rhs_local_relative_humidity"] = df["rhs_local_relative_humidity"].fillna(0)

print("Limited filling applied")

Limited filling applied


In [35]:
# =========================
# 5. FINAL MISSING CHECK
# =========================

print("\n===== FINAL MISSING =====")
print(df[value_cols].isna().mean())


===== FINAL MISSING =====
ps_pressure                    0.000827
rhs_local_relative_humidity    0.000000
rhs_humidity_local_temp        0.000817
dtype: float64


In [36]:
# =========================
# 6. SIGNAL DISTORTION CHECK
# =========================

print("\n===== SIGNAL CHECK =====")

for col in value_cols:
    original_missing = df[f"{col}_was_missing"] == 1
    
    std_real = df.loc[~original_missing, col].std()
    std_filled = df.loc[original_missing, col].std()
    
    print(f"\n{col}:")
    print("STD (real):", std_real)
    print("STD (filled):", std_filled)


===== SIGNAL CHECK =====

ps_pressure:
STD (real): 32.8683480142121
STD (filled): 34.49787591254798

rhs_local_relative_humidity:
STD (real): 4.848303742952296
STD (filled): 8.068605236815914

rhs_humidity_local_temp:
STD (real): 23.200826462002993
STD (filled): 9.557329110839941


In [37]:
# =========================
# 7. FINAL DATASET
# =========================

final_cols = [
    # Time
    "continuous_time",
    "segment_id",

    # Target + features
    "ps_pressure",
    "rhs_local_relative_humidity",
    "rhs_humidity_local_temp",

    # Time features
    "seconds_in_sol",
    "hour_of_sol",
    "sin_hour",
    "cos_hour",
    "is_day",

    # Seasonal features
    "ls_deg",
    "sin_ls",
    "cos_ls",

    # Missing flags
    "ps_pressure_was_missing",
    "rhs_local_relative_humidity_was_missing",
    "rhs_humidity_local_temp_was_missing"
]

df = df[final_cols]

print("\nFinal shape:", df.shape)


Final shape: (107678, 16)


In [38]:
# =========================
# 8. SAVE
# =========================

df.to_parquet(OUT_PATH, index=False)

print("\nSaved:", OUT_PATH)


Saved: C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_clean_STEP3.parquet


In [39]:
# STEP 4 — Chronological SPLIT

In [40]:
# =========================
# STEP 4A — CHRONOLOGICAL SPLIT (FINAL)
# =========================

import pandas as pd
from pathlib import Path

# -------------------------
# PATHS
# -------------------------
DATA_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_clean_STEP3.parquet")
OUT_DIR   = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_chrono")
OUT_DIR.mkdir(exist_ok=True)

In [41]:
# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_parquet(DATA_PATH)

print("Loaded:", df.shape)

Loaded: (107678, 16)


In [42]:
# =========================
# 1. SORT BY TIME
# =========================
df = df.sort_values("continuous_time").reset_index(drop=True)

print("\nTime monotonic:", df["continuous_time"].is_monotonic_increasing)

# =========================
# 2. SPLIT (60 / 20 / 20)
# =========================
N = len(df)

train_end = int(0.6 * N)
val_end   = int(0.8 * N)

df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()


Time monotonic: True


In [43]:
# =========================
# 3. SANITY CHECK
# =========================

print("\n===== SPLIT SIZES =====")
print("Train:", df_train.shape)
print("Val  :", df_val.shape)
print("Test :", df_test.shape)

assert df_train["continuous_time"].max() < df_val["continuous_time"].min()
assert df_val["continuous_time"].max() < df_test["continuous_time"].min()

print("\n Chronological order verified (NO LEAKAGE)")


===== SPLIT SIZES =====
Train: (64606, 16)
Val  : (21536, 16)
Test : (21536, 16)

 Chronological order verified (NO LEAKAGE)


In [44]:
# =========================
# 4. TIME RANGE CHECK
# =========================

print("\n===== TIME RANGE =====")

print("Train:",
      df_train["continuous_time"].min(),
      "→",
      df_train["continuous_time"].max())

print("Val:",
      df_val["continuous_time"].min(),
      "→",
      df_val["continuous_time"].max())

print("Test:",
      df_test["continuous_time"].min(),
      "→",
      df_test["continuous_time"].max())


===== TIME RANGE =====
Train: 26632560.0 → 32157180.0
Val: 32157240.0 → 34377720.0
Test: 34377780.0 → 37283220.0


In [45]:
# =========================
# 5. Ls DISTRIBUTION (ANALYSIS ONLY)
# =========================

print("\n===== Ls DISTRIBUTION =====")

print("\nTrain Ls:")
print(df_train["ls_deg"].describe())

print("\nVal Ls:")
print(df_val["ls_deg"].describe())

print("\nTest Ls:")
print(df_test["ls_deg"].describe())


===== Ls DISTRIBUTION =====

Train Ls:
count    64606.000000
mean       177.966212
std        104.335526
min          0.020000
25%         82.402500
50%        155.780000
75%        277.020000
max        359.970000
Name: ls_deg, dtype: float64

Val Ls:
count    21536.000000
mean       175.987618
std         91.091933
min          0.320000
25%         92.577500
50%        159.425000
75%        264.160000
max        359.990000
Name: ls_deg, dtype: float64

Test Ls:
count    21536.000000
mean       176.610671
std         83.660088
min          0.020000
25%         98.980000
50%        162.185000
75%        258.672500
max        359.890000
Name: ls_deg, dtype: float64


In [46]:
# =========================
# 6. SEGMENT CHECK
# =========================

def check_segments(df_split, name):
    seg_sizes = df_split.groupby("segment_id").size()
    
    small_segments = (seg_sizes < 5).sum()
    
    print(f"\n{name} segment check:")
    print("Total segments:", df_split["segment_id"].nunique())
    print("Small segments (<5 rows):", small_segments)

check_segments(df_train, "TRAIN")
check_segments(df_val, "VAL")
check_segments(df_test, "TEST")


TRAIN segment check:
Total segments: 567
Small segments (<5 rows): 0

VAL segment check:
Total segments: 237
Small segments (<5 rows): 0

TEST segment check:
Total segments: 299
Small segments (<5 rows): 1


In [47]:
# =========================
# 7. SAVE SPLITS
# =========================

train_path = OUT_DIR / "train.parquet"
val_path   = OUT_DIR / "val.parquet"
test_path  = OUT_DIR / "test.parquet"

df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)
df_test.to_parquet(test_path, index=False)

print("\nSaved splits:")
print("Train:", train_path)
print("Val  :", val_path)
print("Test :", test_path)


Saved splits:
Train: C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_chrono\train.parquet
Val  : C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_chrono\val.parquet
Test : C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_chrono\test.parquet


In [48]:
# =========================
# STEP 5B — Ls BLOCK SPLIT (SECONDARY)
# =========================

import pandas as pd
from pathlib import Path

# -------------------------
# PATHS (NEW FOLDER)
# -------------------------
DATA_PATH = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_final\meda_clean_STEP3.parquet")

OUT_DIR = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_ls")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(DATA_PATH)

print("Loaded:", df.shape)

# =========================
# SORT (CRITICAL)
# =========================
df = df.sort_values("continuous_time").reset_index(drop=True)

# =========================
# DEFINE Ls SPLITS
# =========================

def in_range(x, low, high):
    return (x >= low) & (x < high)

train_mask = (
    in_range(df["ls_deg"], 10, 110) |
    in_range(df["ls_deg"], 200, 260)
)

val_mask = in_range(df["ls_deg"], 110, 200)
test_mask = in_range(df["ls_deg"], 260, 360)

# =========================
# APPLY SPLITS
# =========================

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print("\n===== SPLIT SIZES =====")
print("Train:", df_train.shape)
print("Val  :", df_val.shape)
print("Test :", df_test.shape)

# =========================
# GAP SAFETY CHECK
# =========================

def check_segments(df_split, name):
    seg_sizes = df_split.groupby("segment_id").size()
    
    print(f"\n{name}:")
    print("Total segments:", seg_sizes.shape[0])
    print("Small segments (<5 rows):", (seg_sizes < 5).sum())
    print("Median segment size:", seg_sizes.median())

check_segments(df_train, "TRAIN")
check_segments(df_val, "VAL")
check_segments(df_test, "TEST")

# =========================
# REMOVE VERY SMALL SEGMENTS
# =========================

def remove_small_segments(df_split, min_size=5):
    seg_sizes = df_split.groupby("segment_id").size()
    valid_segments = seg_sizes[seg_sizes >= min_size].index
    return df_split[df_split["segment_id"].isin(valid_segments)]

df_train = remove_small_segments(df_train)
df_val   = remove_small_segments(df_val)
df_test  = remove_small_segments(df_test)

# =========================
# FINAL CHECK — Ls DISTRIBUTION
# =========================

print("\n===== Ls DISTRIBUTION =====")

print("\nTrain Ls:")
print(df_train["ls_deg"].describe())

print("\nVal Ls:")
print(df_val["ls_deg"].describe())

print("\nTest Ls:")
print(df_test["ls_deg"].describe())

# =========================
# IMPORTANT — CONTIGUITY CHECK
# =========================

def check_time_continuity(df_split, name):
    dt = df_split["continuous_time"].diff()
    large_gaps = (dt > (10 * dt.median())).sum()
    
    print(f"\n{name} time check:")
    print("Large gaps:", large_gaps)

check_time_continuity(df_train, "TRAIN")
check_time_continuity(df_val, "VAL")
check_time_continuity(df_test, "TEST")

# =========================
# SAVE (SEPARATE FROM CHRONO)
# =========================

train_path = OUT_DIR / "train_ls.parquet"
val_path   = OUT_DIR / "val_ls.parquet"
test_path  = OUT_DIR / "test_ls.parquet"

df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)
df_test.to_parquet(test_path, index=False)

print("\nSaved Ls splits:")
print("Train:", train_path)
print("Val  :", val_path)
print("Test :", test_path)

Loaded: (107678, 16)

===== SPLIT SIZES =====
Train: (56692, 16)
Val  : (13904, 16)
Test : (36088, 16)

TRAIN:
Total segments: 672
Small segments (<5 rows): 8
Median segment size: 66.0

VAL:
Total segments: 226
Small segments (<5 rows): 10
Median segment size: 56.0

TEST:
Total segments: 464
Small segments (<5 rows): 1
Median segment size: 66.0

===== Ls DISTRIBUTION =====

Train Ls:
count    56674.000000
mean       118.236011
std         73.755176
min         10.010000
25%         73.300000
50%         90.810000
75%        201.087500
max        259.990000
Name: ls_deg, dtype: float64

Val Ls:
count    13881.000000
mean       140.834236
std         25.481169
min        110.000000
25%        119.130000
50%        133.330000
75%        159.050000
max        199.980000
Name: ls_deg, dtype: float64

Test Ls:
count    36086.000000
mean       288.813662
std         24.085651
min        260.000000
25%        270.930000
50%        281.380000
75%        299.990000
max        359.990000
Name: ls

In [49]:
# =========================
# COMPARE FEATURES — CHRONO vs LS
# =========================

import pandas as pd
from pathlib import Path

# ---- PATHS ----
chrono_path = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_chrono\train.parquet")
ls_path     = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_ls\train_ls.parquet")

df_chrono = pd.read_parquet(chrono_path)
df_ls     = pd.read_parquet(ls_path)

# -------------------------
# COLUMN LIST
# -------------------------
cols_chrono = set(df_chrono.columns)
cols_ls     = set(df_ls.columns)

print("===== COLUMN COUNT =====")
print("Chrono:", len(cols_chrono))
print("LS    :", len(cols_ls))

# -------------------------
# DIFFERENCE CHECK
# -------------------------
only_in_chrono = cols_chrono - cols_ls
only_in_ls     = cols_ls - cols_chrono

print("\n===== DIFFERENCES =====")
print("Only in CHRONO:", only_in_chrono)
print("Only in LS    :", only_in_ls)

# -------------------------
# COMMON FEATURES
# -------------------------
common = cols_chrono & cols_ls

print("\n===== COMMON FEATURES =====")
for c in sorted(common):
    print(c)

# -------------------------
# ORDER CHECK (important for modeling)
# -------------------------
print("\n===== COLUMN ORDER CHECK =====")
print("Chrono first 10:", list(df_chrono.columns[:10]))
print("LS first 10    :", list(df_ls.columns[:10]))

===== COLUMN COUNT =====
Chrono: 16
LS    : 16

===== DIFFERENCES =====
Only in CHRONO: set()
Only in LS    : set()

===== COMMON FEATURES =====
continuous_time
cos_hour
cos_ls
hour_of_sol
is_day
ls_deg
ps_pressure
ps_pressure_was_missing
rhs_humidity_local_temp
rhs_humidity_local_temp_was_missing
rhs_local_relative_humidity
rhs_local_relative_humidity_was_missing
seconds_in_sol
segment_id
sin_hour
sin_ls

===== COLUMN ORDER CHECK =====
Chrono first 10: ['continuous_time', 'segment_id', 'ps_pressure', 'rhs_local_relative_humidity', 'rhs_humidity_local_temp', 'seconds_in_sol', 'hour_of_sol', 'sin_hour', 'cos_hour', 'is_day']
LS first 10    : ['continuous_time', 'segment_id', 'ps_pressure', 'rhs_local_relative_humidity', 'rhs_humidity_local_temp', 'seconds_in_sol', 'hour_of_sol', 'sin_hour', 'cos_hour', 'is_day']


In [135]:
# =========================
# STEP 5C — WINDOWING (CHRONO FOCUSED)
# =========================

In [136]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIG
# =========================
TARGET_COL = "ps_pressure"

BASE_FEATURE_COLS = [
    "ps_pressure",
    "rhs_local_relative_humidity",
    "rhs_humidity_local_temp",
    "sin_hour",
    "cos_hour",
    "sin_ls",
    "cos_ls",
    "is_day"
]

LOOKBACK = 360  # 2 hours

# FULL MULTI-HORIZON (ONLY FOR CHRONO)
HORIZONS = [60, 180, 360, 720, 1440]

In [137]:
# =========================
# WINDOW FUNCTION
# =========================
def create_windows(df, name):

    print(f"\n===== {name} =====")

    df = df.sort_values("continuous_time").reset_index(drop=True)

    # -------------------------
    # DELTA FEATURE
    # -------------------------
    df["delta_p_prev"] = df["ps_pressure"].diff()

    if "segment_id" in df.columns:
        segment_change = df["segment_id"].diff().fillna(0) != 0
        df.loc[segment_change, "delta_p_prev"] = 0

    df["delta_p_prev"] = df["delta_p_prev"].fillna(0)

    FEATURE_COLS = BASE_FEATURE_COLS + ["delta_p_prev"]

    print("Input rows:", len(df))
    print("Horizons:", HORIZONS)

    values = df[FEATURE_COLS].values.astype(np.float32)
    target = df[TARGET_COL].values.astype(np.float32)

    valid_mask = ~df[FEATURE_COLS].isna().any(axis=1)

    # -------------------------
    # GAP DETECTION
    # -------------------------
    dt = df["continuous_time"].diff()
    median_dt = dt.median()
    gap_flag = (dt > (10 * median_dt)).fillna(False).astype(int).values

    # -------------------------
    # STORAGE
    # -------------------------
    X, y_delta, y_true, p_last = [], [], [], []

    max_h = max(HORIZONS)

    # -------------------------
    # MAIN LOOP
    # -------------------------
    for i in range(LOOKBACK, len(df) - max_h):

        start = i - LOOKBACK
        end = i

        # RELAXED (important for 24h)
        if valid_mask[start:end].mean() < 0.95:
            continue

        # RELAXED GAP CONTROL (critical fix)
        if gap_flag[start + 1 : i + max_h].sum() > 20:
            continue

        future_idx = [i + h - 1 for h in HORIZONS]
        if np.isnan(target[future_idx]).any():
            continue

        x = values[start:end]

        if np.isnan(x).any() or np.isinf(x).any():
            continue

        last_p = target[i - 1]

        if np.isnan(last_p) or np.isinf(last_p):
            continue

        # MULTI-HORIZON TARGETS
        y_d = np.array(
            [target[i + h - 1] - last_p for h in HORIZONS],
            dtype=np.float32
        )

        y_t = np.array(
            [target[i + h - 1] for h in HORIZONS],
            dtype=np.float32
        )

        if np.isnan(y_d).any():
            continue

        X.append(x)
        y_delta.append(y_d)
        y_true.append(y_t)
        p_last.append(last_p)

    X = np.array(X, dtype=np.float32)
    y_delta = np.array(y_delta, dtype=np.float32)
    y_true = np.array(y_true, dtype=np.float32)
    p_last = np.array(p_last, dtype=np.float32)

    print("Windows shape:", X.shape)

    if len(X) == 0:
        raise ValueError(f"{name}: NO WINDOWS CREATED")

    return X, y_delta, y_true, p_last

In [138]:
# =========================
# PATHS
# =========================
CHRONO_DIR = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_split_chrono")
OUT_DIR = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_windows")
OUT_DIR.mkdir(exist_ok=True)


# =========================
# LOAD DATA
# =========================
train = pd.read_parquet(CHRONO_DIR / "train.parquet")
val   = pd.read_parquet(CHRONO_DIR / "val.parquet")
test  = pd.read_parquet(CHRONO_DIR / "test.parquet")


# =========================
# CREATE WINDOWS
# =========================
X_train, y_train_d, y_train_t, p_train = create_windows(train, "CHRONO TRAIN")
X_val,   y_val_d,   y_val_t,   p_val   = create_windows(val,   "CHRONO VAL")
X_test,  y_test_d,  y_test_t,  p_test  = create_windows(test,  "CHRONO TEST")


===== CHRONO TRAIN =====
Input rows: 64606
Horizons: [60, 180, 360, 720, 1440]
Windows shape: (44393, 360, 9)

===== CHRONO VAL =====
Input rows: 21536
Horizons: [60, 180, 360, 720, 1440]
Windows shape: (9779, 360, 9)

===== CHRONO TEST =====
Input rows: 21536
Horizons: [60, 180, 360, 720, 1440]
Windows shape: (4443, 360, 9)


In [139]:
# =========================
# SAVE
# =========================
np.savez_compressed(
    OUT_DIR / "windows_chrono_multi.npz",
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_train_delta=y_train_d,
    y_val_delta=y_val_d,
    y_test_delta=y_test_d,
    y_train_true=y_train_t,
    y_val_true=y_val_t,
    y_test_true=y_test_t,
    p_last_train=p_train,
    p_last_val=p_val,
    p_last_test=p_test
)

print("\nSaved CHRONO multi-horizon windows")
print("\n===== STEP 5C COMPLETE =====")


Saved CHRONO multi-horizon windows

===== STEP 5C COMPLETE =====


In [140]:
# =========================
# STEP 5C SANITY CHECK (WINDOW LEVEL)
# =========================

import numpy as np

# ---- LOAD YOUR WINDOWS ----
data = np.load(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_windows\windows_chrono_multi.npz")

X_train = data["X_train"]
y_train = data["y_train_delta"]

print("X shape:", X_train.shape)
print("y shape:", y_train.shape)

# =========================
# 1. BASIC CHECK
# =========================
print("\n===== BASIC =====")
print("Timesteps:", X_train.shape[1])
print("Features :", X_train.shape[2])

# =========================
# 2. NAN CHECK
# =========================
print("\n===== NAN CHECK =====")
print("NaN in X:", np.isnan(X_train).sum())
print("NaN in y:", np.isnan(y_train).sum())

# =========================
# 3. INF CHECK
# =========================
print("\n===== INF CHECK =====")
print("Inf in X:", np.isinf(X_train).sum())
print("Inf in y:", np.isinf(y_train).sum())

# =========================
# 4. VALUE RANGE CHECK
# =========================
print("\n===== VALUE RANGE =====")

print("Min:", np.min(X_train))
print("Max:", np.max(X_train))
print("Mean:", np.mean(X_train))
print("Std :", np.std(X_train))

# =========================
# 5. FEATURE-WISE CHECK
# =========================
print("\n===== FEATURE STATS =====")

for i in range(X_train.shape[2]):
    f = X_train[:, :, i]
    print(f"\nFeature {i}:")
    print("  min :", np.min(f))
    print("  max :", np.max(f))
    print("  mean:", np.mean(f))
    print("  std :", np.std(f))

# =========================
# 6. DELTA FEATURE CHECK (LAST FEATURE)
# =========================
print("\n===== DELTA FEATURE CHECK =====")

delta_feature = X_train[:, :, -1]

print("Delta min:", np.min(delta_feature))
print("Delta max:", np.max(delta_feature))
print("Delta mean:", np.mean(delta_feature))
print("Delta std:", np.std(delta_feature))

# =========================
# 7. TARGET CHECK
# =========================
print("\n===== TARGET CHECK =====")

print("y min:", np.min(y_train))
print("y max:", np.max(y_train))
print("y mean:", np.mean(y_train))
print("y std:", np.std(y_train))

print("\n===== SANITY CHECK DONE =====")

X shape: (44393, 360, 9)
y shape: (44393, 5)

===== BASIC =====
Timesteps: 360
Features : 9

===== NAN CHECK =====
NaN in X: 0
NaN in y: 0

===== INF CHECK =====
Inf in X: 0
Inf in y: 0

===== VALUE RANGE =====
Min: -10.03078
Max: 674.90405
Mean: 96.38409
Std : 203.00856

===== FEATURE STATS =====

Feature 0:
  min : 585.751
  max : 674.90405
  mean: 633.3899
  std : 12.650454

Feature 1:
  min : 0.0
  max : 42.686
  mean: 4.213334
  std : 6.6489406

Feature 2:
  min : 195.706
  max : 278.32227
  mean: 229.21078
  std : 24.0116

Feature 3:
  min : -1.0
  max : 1.0
  mean: -0.0155960675
  std : 0.70819175

Feature 4:
  min : -1.0
  max : 1.0
  mean: 0.00623731
  std : 0.7058199

Feature 5:
  min : -1.0
  max : 1.0
  mean: 0.033956055
  std : 0.8687833

Feature 6:
  min : -0.9999999
  max : 0.99999994
  mean: 0.12602985
  std : 0.47768146

Feature 7:
  min : 0.0
  max : 1.0
  mean: 0.48917717
  std : 0.49988112

Feature 8:
  min : -10.03078
  max : 4.9024787
  mean: 0.0034718984
  std : 

In [132]:
# =========================
# STEP 5D — SCALING
# =========================

In [141]:
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib

# =========================
# PATHS
# =========================
WINDOW_DIR = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\data_windows")
OUT_DIR    = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\New_data_scaled")
OUT_DIR.mkdir(exist_ok=True)

In [142]:
# =========================
# LOAD WINDOW DATA
# =========================
chrono = np.load(WINDOW_DIR / "windows_chrono_multi.npz", allow_pickle=True)

# -------------------------
# EXTRACT CHRONO
# -------------------------
X_train = chrono["X_train"]
X_val   = chrono["X_val"]
X_test  = chrono["X_test"]

y_train = chrono["y_train_delta"]
y_val   = chrono["y_val_delta"]
y_test  = chrono["y_test_delta"]

# IMPORTANT (for reconstruction later)
y_train_true = chrono["y_train_true"]
y_val_true   = chrono["y_val_true"]
y_test_true  = chrono["y_test_true"]

p_last_train = chrono["p_last_train"]
p_last_val   = chrono["p_last_val"]
p_last_test  = chrono["p_last_test"]

print("Loaded CHRONO:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

Loaded CHRONO:
X_train: (44393, 360, 9)
y_train: (44393, 5)


In [143]:
# =========================
# 1. SCALE INPUTS (X)
# =========================
N_train, T, F = X_train.shape

x_scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, F)
x_scaler.fit(X_train_2d)

print("\n X scaler fitted on TRAIN only")

def scale_X(X, name):
    N, T, F = X.shape
    X_2d = X.reshape(-1, F)
    X_scaled = x_scaler.transform(X_2d)
    X_scaled = X_scaled.reshape(N, T, F)
    print(f"{name} scaled:", X_scaled.shape)
    return X_scaled

X_train_scaled = scale_X(X_train, "Train")
X_val_scaled   = scale_X(X_val,   "Val")
X_test_scaled  = scale_X(X_test,  "Test")


 X scaler fitted on TRAIN only
Train scaled: (44393, 360, 9)
Val scaled: (9779, 360, 9)
Test scaled: (4443, 360, 9)


In [144]:
# =========================
# 2. SCALE TARGETS
# =========================
y_scaler = StandardScaler()

y_scaler.fit(y_train)

print("\n y scaler fitted on TRAIN only")

def scale_y(y, name):
    y_scaled = y_scaler.transform(y)
    print(f"{name} y scaled:", y_scaled.shape)
    return y_scaled

y_train_scaled = scale_y(y_train, "Train")
y_val_scaled   = scale_y(y_val,   "Val")
y_test_scaled  = scale_y(y_test,  "Test")


 y scaler fitted on TRAIN only
Train y scaled: (44393, 5)
Val y scaled: (9779, 5)
Test y scaled: (4443, 5)


In [145]:
# =========================
# SANITY CHECK
# =========================
print("\n===== SANITY CHECK =====")

print("X train mean:", np.mean(X_train_scaled))
print("X train std :", np.std(X_train_scaled))

print("y train mean:", np.mean(y_train_scaled))
print("y train std :", np.std(y_train_scaled))

print("NaN check X:", np.isnan(X_train_scaled).sum())
print("NaN check y:", np.isnan(y_train_scaled).sum())


===== SANITY CHECK =====
X train mean: -2.625007e-09
X train std : 1.0000004
y train mean: 8.111806e-09
y train std : 1.0
NaN check X: 0
NaN check y: 0


In [146]:
# =========================
# SAVE CHRONO (FULL PACKAGE)
# =========================
np.savez_compressed(
    OUT_DIR / "scaled_chrono.npz",

    # Inputs
    X_train=X_train_scaled,
    X_val=X_val_scaled,
    X_test=X_test_scaled,

    # Targets (scaled)
    y_train=y_train_scaled,
    y_val=y_val_scaled,
    y_test=y_test_scaled,

    # Targets (absolute, unscaled)
    y_train_true=y_train_true,
    y_val_true=y_val_true,
    y_test_true=y_test_true,

    # Last pressure (for reconstruction)
    p_last_train=p_last_train,
    p_last_val=p_last_val,
    p_last_test=p_last_test
)

print("\n Saved scaled CHRONO dataset")


 Saved scaled CHRONO dataset


In [147]:
# =========================
# SAVE SCALERS
# =========================
joblib.dump(x_scaler, OUT_DIR / "x_scaler.pkl")
joblib.dump(y_scaler, OUT_DIR / "y_scaler.pkl")

print(" Scalers saved")

 Scalers saved


In [149]:
# =========================
# STEP 6A — BASELINES (5-HORIZON)
# =========================

In [150]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import joblib

# =========================
# PATHS
# =========================
DATA_DIR = Path(r"C:\Users\thari\OneDrive\Desktop\NASA_MEDA\New_data_scaled")
Y_SCALER_PATH = DATA_DIR / "y_scaler.pkl"

In [151]:
# =========================
# LOAD DATA
# =========================
data = np.load(DATA_DIR / "scaled_chrono.npz", allow_pickle=True)
y_scaler = joblib.load(Y_SCALER_PATH)

X_train = data["X_train"]
X_test  = data["X_test"]

y_train = data["y_train"]
y_test  = data["y_test"]

y_test_true = data["y_test_true"]
p_last_test = data["p_last_test"]

print("Loaded data:")
print("Train:", X_train.shape)

Loaded data:
Train: (44393, 360, 9)


In [152]:
# =========================
# CONFIG
# =========================
HORIZON_LABELS = ["1h", "3h", "6h", "12h", "24h"]

# =========================
# HELPERS
# =========================

def flatten(X):
    return X.reshape(X.shape[0], -1)

def delta_to_absolute(p_last, y_delta):
    return p_last[:, None] + y_delta

def compute_metrics(y_true, y_pred, name):
    results = {}

    for i, h in enumerate(HORIZON_LABELS):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))

        results[f"{name}_MAE_{h}"] = round(mae, 3)
        results[f"{name}_RMSE_{h}"] = round(rmse, 3)

    return results

In [153]:
# =========================
# PREPARE DATA
# =========================

X_train_flat = flatten(X_train)
X_test_flat  = flatten(X_test)

y_test_abs = y_test_true

In [154]:
# =========================
# BASELINE 1 — PERSISTENCE
# =========================

# Repeat last pressure across 5 horizons
y_pred_persist_abs = np.repeat(
    p_last_test[:, None],
    len(HORIZON_LABELS),
    axis=1
)

results_persist = compute_metrics(y_test_abs, y_pred_persist_abs, "Persistence")

In [155]:
# =========================
# BASELINE 2 — LINEAR REGRESSION
# =========================

model_lr = LinearRegression()
model_lr.fit(X_train_flat, y_train)

y_pred_lr_scaled = model_lr.predict(X_test_flat)

y_pred_lr_delta = y_scaler.inverse_transform(y_pred_lr_scaled)
y_pred_lr_abs = delta_to_absolute(p_last_test, y_pred_lr_delta)

results_lr = compute_metrics(y_test_abs, y_pred_lr_abs, "LinearRegression")

In [156]:
# =========================
# BASELINE 3 — RANDOM FOREST
# =========================

model_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=18,
    n_jobs=-1,
    random_state=42
)

model_rf.fit(X_train_flat, y_train)

y_pred_rf_scaled = model_rf.predict(X_test_flat)

y_pred_rf_delta = y_scaler.inverse_transform(y_pred_rf_scaled)
y_pred_rf_abs = delta_to_absolute(p_last_test, y_pred_rf_delta)

results_rf = compute_metrics(y_test_abs, y_pred_rf_abs, "RandomForest")

In [157]:
# =========================
# PRINT RESULTS
# =========================

print("\n===== BASELINE RESULTS (5-HORIZON) =====")

all_results = {}
all_results.update(results_persist)
all_results.update(results_lr)
all_results.update(results_rf)

for k, v in all_results.items():
    print(f"{k}: {v}")


===== BASELINE RESULTS (5-HORIZON) =====
Persistence_MAE_1h: 6.605999946594238
Persistence_RMSE_1h: 9.11299991607666
Persistence_MAE_3h: 13.90999984741211
Persistence_RMSE_3h: 17.17099952697754
Persistence_MAE_6h: 17.63599967956543
Persistence_RMSE_6h: 21.51799964904785
Persistence_MAE_12h: 15.965999603271484
Persistence_RMSE_12h: 19.139999389648438
Persistence_MAE_24h: 15.085000038146973
Persistence_RMSE_24h: 19.024999618530273
LinearRegression_MAE_1h: 11.873000144958496
LinearRegression_RMSE_1h: 13.866999626159668
LinearRegression_MAE_3h: 38.57600021362305
LinearRegression_RMSE_3h: 40.819000244140625
LinearRegression_MAE_6h: 42.691001892089844
LinearRegression_RMSE_6h: 44.262001037597656
LinearRegression_MAE_12h: 12.664999961853027
LinearRegression_RMSE_12h: 15.241000175476074
LinearRegression_MAE_24h: 51.04800033569336
LinearRegression_RMSE_24h: 52.5890007019043
RandomForest_MAE_1h: 5.408
RandomForest_RMSE_1h: 7.539
RandomForest_MAE_3h: 9.236
RandomForest_RMSE_3h: 12.911
RandomFore